# Practical 1: Parsing Text, CSV, HTML, XML & JSON, Binary Files, Regular Expressions and SQL CRUD

|  |  |
|---|---|
| **Name** | Aayushman Verma |
| **Roll No.** | 57 |
| **College** | B. K. Birla College, Kalyan |
| **Student ID** | 5176202 |

**Aim:** To write Python programs that read text, CSV, HTML, XML and JSON documents, extract data from them and check the data for missing values, duplicates and out-of-range (anomalous) values; to read and write binary files; to search, split and replace text using regular expressions; and to design a relational database and perform CRUD operations with SQL.

## Task 1: Create the Employee Data Files (Text, CSV, HTML, XML, JSON)

In [1]:
# Name: Ayushman Rajeev Verma
# Practical 1 - Setup: one employee data set stored in five different file formats
print("Name: Ayushman Rajeev Verma")
import json

EMP = [  # (emp_id, name, dept, salary, age)  -> contains deliberate data problems
    ("E101", "Aarav Mehta", "IT", 65000, 28),
    ("E102", "Isha Nair", "HR", 48000, 31),
    ("E103", "Kabir Singh", "IT", -52000, 45),        # negative salary
    ("E104", "Meera Joshi", "", 39000, 26),           # missing department
    ("E105", "Nikhil Rao", "Finance", 72000, 150),    # impossible age
    ("E102", "Isha Nair", "HR", 48000, 31),           # duplicate record
    ("E106", "Pooja Desai", "Sales", None, 35),       # missing salary
    ("E107", "Sameer Khan", "Sales", 41000, 17),      # below working age
]
show = lambda v: "" if v is None else str(v)

with open("employees.txt", "w") as f:                 # pipe-delimited text
    f.write("# Employee export | delimiter = |\n")
    for rec in EMP:
        f.write("|".join(show(v) for v in rec) + "\n")

with open("employees.csv", "w") as f:                 # comma separated
    f.write("emp_id,name,dept,salary,age\n")
    for rec in EMP:
        f.write(",".join(show(v) for v in rec) + "\n")

rows = "".join("<tr>" + "".join(f"<td>{show(v)}</td>" for v in rec) + "</tr>\n" for rec in EMP)
with open("employees.html", "w") as f:                # HTML page with a table
    f.write("<html><body><h1>Employee Directory</h1>\n<table border='1'>\n"
            "<tr><th>emp_id</th><th>name</th><th>dept</th><th>salary</th><th>age</th></tr>\n" + rows + "</table></body></html>")

with open("employees.xml", "w") as f:                 # XML where the data sits in attributes
    f.write("<company>\n")
    for e, n, d, s, a in EMP:
        f.write(f'  <employee emp_id="{e}" name="{n}" dept="{d}" salary="{show(s)}" age="{a}"/>\n')
    f.write("</company>")

records = [dict(zip(["emp_id", "name", "dept", "salary", "age"], rec)) for rec in EMP]
for r in records:
    r["skills"] = ["python", "sql"] if r["dept"] == "IT" else ["excel"]
with open("employees.json", "w") as f:
    json.dump({"company": "Nova Tech", "employees": records}, f, indent=2)

print("Created: employees.txt, employees.csv, employees.html, employees.xml, employees.json")

Name: Ayushman Rajeev Verma
Created: employees.txt, employees.csv, employees.html, employees.xml, employees.json


## Task 2: A Reusable Data Quality Checker Class

One class is used for every file format. It converts empty values to missing values, fixes data types and reports missing values, duplicate rows and values outside the allowed range (salary above 0 and age between 18 and 65).

In [2]:
# Name: Ayushman Rajeev Verma
import pandas as pd
import numpy as np

class DataQualityChecker:
    """Reusable quality checker: missing values, duplicates and out-of-range values."""
    RULES = {"salary": (1, 1_000_000), "age": (18, 65)}      # allowed (min, max)

    def __init__(self, df, source):
        self.source = source
        self.df = df.replace("", np.nan)                      # treat empty strings as missing
        for col in ("salary", "age"):
            self.df[col] = pd.to_numeric(self.df[col], errors="coerce")

    def missing(self):
        return self.df.isna().sum()

    def duplicates(self):
        return self.df[self.df.duplicated(keep="first")]

    def anomalies(self):
        found = []
        for col, (lo, hi) in self.RULES.items():
            bad = self.df[(self.df[col] < lo) | (self.df[col] > hi)]
            found += [(r.emp_id, col, r[col]) for _, r in bad.iterrows()]
        return found

    def report(self):
        print(f"##### QUALITY REPORT : {self.source} #####")
        print(self.df.to_string(index=False), "\n")
        miss = self.missing()
        print("Missing values :", {c: int(v) for c, v in miss.items() if v})
        dup = self.duplicates()
        print("Duplicate rows :", len(dup), list(dup["emp_id"]))
        print("Out-of-range   :")
        for emp, col, val in self.anomalies():
            print(f"   {emp} -> {col} = {val:g}  (allowed {self.RULES[col][0]} to {self.RULES[col][1]})")
        return self.df

## Task 3: Parse a Text File (pipe delimited)

In [3]:
print("Name: Ayushman Rajeev Verma")
def load_txt(path):
    rows = []
    with open(path) as fh:
        for line in fh:
            if line.startswith("#") or not line.strip():
                continue
            rows.append(line.rstrip("\n").split("|"))
    return pd.DataFrame(rows, columns=["emp_id", "name", "dept", "salary", "age"])

DataQualityChecker(load_txt("employees.txt"), "TEXT (pipe delimited)").report()

Name: Ayushman Rajeev Verma
##### QUALITY REPORT : TEXT (pipe delimited) #####
emp_id        name    dept   salary  age
  E101 Aarav Mehta      IT  65000.0   28
  E102   Isha Nair      HR  48000.0   31
  E103 Kabir Singh      IT -52000.0   45
  E104 Meera Joshi     NaN  39000.0   26
  E105  Nikhil Rao Finance  72000.0  150
  E102   Isha Nair      HR  48000.0   31
  E106 Pooja Desai   Sales      NaN   35
  E107 Sameer Khan   Sales  41000.0   17 

Missing values : {'dept': 1, 'salary': 1}
Duplicate rows : 1 ['E102']
Out-of-range   :
   E103 -> salary = -52000  (allowed 1 to 1000000)
   E105 -> age = 150  (allowed 18 to 65)
   E107 -> age = 17  (allowed 18 to 65)


,emp_id,name,dept,salary,age
0,E101,Aarav Mehta,IT,65000.0,28
1,E102,Isha Nair,HR,48000.0,31
2,E103,Kabir Singh,IT,-52000.0,45
3,E104,Meera Joshi,NaN,39000.0,26
4,E105,Nikhil Rao,Finance,72000.0,150
5,E102,Isha Nair,HR,48000.0,31
6,E106,Pooja Desai,Sales,NaN,35
7,E107,Sameer Khan,Sales,41000.0,17


## Task 4: Parse a CSV File

In [4]:
print("Name: Ayushman Rajeev Verma")
df_csv = pd.read_csv("employees.csv", dtype=str, keep_default_na=False)
DataQualityChecker(df_csv, "CSV file").report()

Name: Ayushman Rajeev Verma
##### QUALITY REPORT : CSV file #####
emp_id        name    dept   salary  age
  E101 Aarav Mehta      IT  65000.0   28
  E102   Isha Nair      HR  48000.0   31
  E103 Kabir Singh      IT -52000.0   45
  E104 Meera Joshi     NaN  39000.0   26
  E105  Nikhil Rao Finance  72000.0  150
  E102   Isha Nair      HR  48000.0   31
  E106 Pooja Desai   Sales      NaN   35
  E107 Sameer Khan   Sales  41000.0   17 

Missing values : {'dept': 1, 'salary': 1}
Duplicate rows : 1 ['E102']
Out-of-range   :
   E103 -> salary = -52000  (allowed 1 to 1000000)
   E105 -> age = 150  (allowed 18 to 65)
   E107 -> age = 17  (allowed 18 to 65)


,emp_id,name,dept,salary,age
0,E101,Aarav Mehta,IT,65000.0,28
1,E102,Isha Nair,HR,48000.0,31
2,E103,Kabir Singh,IT,-52000.0,45
3,E104,Meera Joshi,NaN,39000.0,26
4,E105,Nikhil Rao,Finance,72000.0,150
5,E102,Isha Nair,HR,48000.0,31
6,E106,Pooja Desai,Sales,NaN,35
7,E107,Sameer Khan,Sales,41000.0,17


## Task 5: Parse an HTML Document

In [5]:
print("Name: Ayushman Rajeev Verma")
import io
from lxml import html as lh

page = open("employees.html").read()
tree = lh.fromstring(page)
print("Page heading :", tree.xpath("//h1/text()")[0])
print("Table rows   :", len(tree.xpath("//table//tr")) - 1, "\n")

df_html = pd.read_html(io.StringIO(page), keep_default_na=False)[0].astype(str)
DataQualityChecker(df_html, "HTML table").report()

Name: Ayushman Rajeev Verma
Page heading : Employee Directory
Table rows   : 8 

##### QUALITY REPORT : HTML table #####
emp_id        name    dept   salary  age
  E101 Aarav Mehta      IT  65000.0   28
  E102   Isha Nair      HR  48000.0   31
  E103 Kabir Singh      IT -52000.0   45
  E104 Meera Joshi     NaN  39000.0   26
  E105  Nikhil Rao Finance  72000.0  150
  E102   Isha Nair      HR  48000.0   31
  E106 Pooja Desai   Sales      NaN   35
  E107 Sameer Khan   Sales  41000.0   17 

Missing values : {'dept': 1, 'salary': 1}
Duplicate rows : 1 ['E102']
Out-of-range   :
   E103 -> salary = -52000  (allowed 1 to 1000000)
   E105 -> age = 150  (allowed 18 to 65)
   E107 -> age = 17  (allowed 18 to 65)


,emp_id,name,dept,salary,age
0,E101,Aarav Mehta,IT,65000.0,28
1,E102,Isha Nair,HR,48000.0,31
2,E103,Kabir Singh,IT,-52000.0,45
3,E104,Meera Joshi,NaN,39000.0,26
4,E105,Nikhil Rao,Finance,72000.0,150
5,E102,Isha Nair,HR,48000.0,31
6,E106,Pooja Desai,Sales,NaN,35
7,E107,Sameer Khan,Sales,41000.0,17


## Task 6: Parse an XML Document

In [6]:
print("Name: Ayushman Rajeev Verma")
import xml.etree.ElementTree as ET

root = ET.parse("employees.xml").getroot()
df_xml = pd.DataFrame([emp.attrib for emp in root.iter("employee")])
print("Root tag:", root.tag, "| employee elements:", len(df_xml), "\n")
DataQualityChecker(df_xml, "XML (attributes)").report()

Name: Ayushman Rajeev Verma
Root tag: company | employee elements: 8 

##### QUALITY REPORT : XML (attributes) #####
emp_id        name    dept   salary  age
  E101 Aarav Mehta      IT  65000.0   28
  E102   Isha Nair      HR  48000.0   31
  E103 Kabir Singh      IT -52000.0   45
  E104 Meera Joshi     NaN  39000.0   26
  E105  Nikhil Rao Finance  72000.0  150
  E102   Isha Nair      HR  48000.0   31
  E106 Pooja Desai   Sales      NaN   35
  E107 Sameer Khan   Sales  41000.0   17 

Missing values : {'dept': 1, 'salary': 1}
Duplicate rows : 1 ['E102']
Out-of-range   :
   E103 -> salary = -52000  (allowed 1 to 1000000)
   E105 -> age = 150  (allowed 18 to 65)
   E107 -> age = 17  (allowed 18 to 65)


,emp_id,name,dept,salary,age
0,E101,Aarav Mehta,IT,65000.0,28
1,E102,Isha Nair,HR,48000.0,31
2,E103,Kabir Singh,IT,-52000.0,45
3,E104,Meera Joshi,NaN,39000.0,26
4,E105,Nikhil Rao,Finance,72000.0,150
5,E102,Isha Nair,HR,48000.0,31
6,E106,Pooja Desai,Sales,NaN,35
7,E107,Sameer Khan,Sales,41000.0,17


## Task 7: Parse a JSON Document

In [7]:
print("Name: Ayushman Rajeev Verma")
import json

doc = json.load(open("employees.json"))
print("Company:", doc["company"], "| records:", len(doc["employees"]))
df_json = pd.DataFrame(doc["employees"])
df_json["skill_count"] = df_json["skills"].apply(len)       # derived column from a nested list
print("Skill counts:", dict(zip(df_json["emp_id"], df_json["skill_count"])), "\n")
DataQualityChecker(df_json.drop(columns=["skills", "skill_count"]), "JSON document").report()

Name: Ayushman Rajeev Verma
Company: Nova Tech | records: 8
Skill counts: {'E101': 2, 'E102': 1, 'E103': 2, 'E104': 1, 'E105': 1, 'E106': 1, 'E107': 1} 

##### QUALITY REPORT : JSON document #####
emp_id        name    dept   salary  age
  E101 Aarav Mehta      IT  65000.0   28
  E102   Isha Nair      HR  48000.0   31
  E103 Kabir Singh      IT -52000.0   45
  E104 Meera Joshi     NaN  39000.0   26
  E105  Nikhil Rao Finance  72000.0  150
  E102   Isha Nair      HR  48000.0   31
  E106 Pooja Desai   Sales      NaN   35
  E107 Sameer Khan   Sales  41000.0   17 

Missing values : {'dept': 1, 'salary': 1}
Duplicate rows : 1 ['E102']
Out-of-range   :
   E103 -> salary = -52000  (allowed 1 to 1000000)
   E105 -> age = 150  (allowed 18 to 65)
   E107 -> age = 17  (allowed 18 to 65)


,emp_id,name,dept,salary,age
0,E101,Aarav Mehta,IT,65000.0,28
1,E102,Isha Nair,HR,48000.0,31
2,E103,Kabir Singh,IT,-52000.0,45
3,E104,Meera Joshi,NaN,39000.0,26
4,E105,Nikhil Rao,Finance,72000.0,150
5,E102,Isha Nair,HR,48000.0,31
6,E106,Pooja Desai,Sales,NaN,35
7,E107,Sameer Khan,Sales,41000.0,17


## Task 8: Binary Files - Custom File Format with Header and Records

In [8]:
print("Name: Ayushman Rajeev Verma")
import struct, zlib, os

MAGIC, VERSION = b"EMPD", 1
HEADER = struct.Struct("<4sBI")            # magic, version, record count
RECORD = struct.Struct("<4sfB")            # emp code, salary, age

staff = [(b"E101", 65000.0, 28), (b"E102", 48000.0, 31), (b"E106", 39500.5, 35)]
with open("staff.bin", "wb") as f:                         # ---- WRITE
    f.write(HEADER.pack(MAGIC, VERSION, len(staff)))
    for rec in staff:
        f.write(RECORD.pack(*rec))
print("File size:", os.path.getsize("staff.bin"), "bytes  =", HEADER.size, "header +", len(staff), "x", RECORD.size)

data = open("staff.bin", "rb").read()                      # ---- READ (whole file as bytes)
print("First 16 bytes:", data[:16].hex(" "))
magic, ver, count = HEADER.unpack_from(data, 0)
print("Header ->", magic.decode(), "| version", ver, "| records", count)
for i in range(count):
    code, sal, age = RECORD.unpack_from(data, HEADER.size + i * RECORD.size)
    print(f"  record {i+1}: {code.decode()}  salary={sal:.1f}  age={age}")

buf = bytearray(data)                                      # ---- MODIFY in memory, then save
RECORD.pack_into(buf, HEADER.size + RECORD.size, b"E102", 52000.0, 32)
open("staff_updated.bin", "wb").write(buf)
print("Updated record 2 ->", RECORD.unpack_from(open("staff_updated.bin", "rb").read(), HEADER.size + RECORD.size))

print("CRC32 original :", hex(zlib.crc32(data)))           # integrity check
print("CRC32 modified :", hex(zlib.crc32(bytes(buf))))

Name: Ayushman Rajeev Verma
File size: 36 bytes  = 9 header + 3 x 9
First 16 bytes: 45 4d 50 44 01 03 00 00 00 45 31 30 31 00 e8 7d
Header -> EMPD | version 1 | records 3
  record 1: E101  salary=65000.0  age=28
  record 2: E102  salary=48000.0  age=31
  record 3: E106  salary=39500.5  age=35
Updated record 2 -> (b'E102', 52000.0, 32)
CRC32 original : 0xc497c3
CRC32 modified : 0xae62a160


## Task 9: Regular Expressions on a Web-Server Log

In [9]:
print("Name: Ayushman Rajeev Verma")
import re

log = """2026-03-04 10:15:32 INFO  192.168.1.10 GET /home 200
2026-03-04 10:16:05 ERROR 10.0.0.77 POST /login 500
2026-03-04 10:17:44 WARN  172.16.4.9 GET /reports/q1.pdf 404
2026-03-04 10:19:10 ERROR 192.168.1.10 GET /api/data 503"""

# 1. SEARCH with a compiled pattern and named groups
pattern = re.compile(r"(?P<date>\d{4}-\d{2}-\d{2}) (?P<time>[\d:]{8}) (?P<level>[A-Z]+)\s+(?P<ip>[\d.]+) (?P<method>GET|POST) (?P<url>\S+) (?P<code>\d{3})")
for m in pattern.finditer(log):
    print(m.group("level").ljust(5), m.group("ip").ljust(13), m.group("url").ljust(16), m.group("code"))

print("Error lines :", len(re.findall(r"^.*ERROR.*$", log, flags=re.M)))
print("Unique IPs  :", sorted(set(re.findall(r"\b(?:\d{1,3}\.){3}\d{1,3}\b", log))))

# 2. SPLIT
first = log.split("\n")[0]
print("Split line  :", re.split(r"\s+", first))
print("Split on / and space:", re.split(r"[/\s]+", "GET /reports/q1.pdf 404"))

# 3. REPLACE (plain text and with a function)
mask = lambda m: ".".join(m.group().split(".")[:2] + ["x", "x"])
print("Masked IPs  :", re.findall(r"\b\d+\.\d+\.x\.x\b", re.sub(r"\b(?:\d{1,3}\.){3}\d{1,3}\b", mask, log)))
print("Levels lower:", re.sub(r"\b(INFO|WARN|ERROR)\b", lambda m: m.group().lower(), first))
print("Dates DD/MM/YYYY:", re.sub(r"(\d{4})-(\d{2})-(\d{2})", r"\3/\2/\1", first)[:10])

Name: Ayushman Rajeev Verma
INFO  192.168.1.10  /home            200
ERROR 10.0.0.77     /login           500
WARN  172.16.4.9    /reports/q1.pdf  404
ERROR 192.168.1.10  /api/data        503
Error lines : 2
Unique IPs  : ['10.0.0.77', '172.16.4.9', '192.168.1.10']
Split line  : ['2026-03-04', '10:15:32', 'INFO', '192.168.1.10', 'GET', '/home', '200']
Split on / and space: ['GET', 'reports', 'q1.pdf', '404']
Masked IPs  : ['192.168.x.x', '10.0.x.x', '172.16.x.x', '192.168.x.x']
Levels lower: 2026-03-04 10:15:32 info  192.168.1.10 GET /home 200
Dates DD/MM/YYYY: 04/03/2026


## Task 10: Relational Database Design (Library System) and CRUD Operations

Three related tables - Book, Member and Issue - are created. Issue links books and members through foreign keys; deleting a book also removes its issue records (ON DELETE CASCADE).

In [10]:
print("Name: Ayushman Rajeev Verma")
import sqlite3

con = sqlite3.connect("library.db")
cur = con.cursor()
cur.executescript("""
PRAGMA foreign_keys = ON;
DROP TABLE IF EXISTS Issue; DROP TABLE IF EXISTS Member; DROP TABLE IF EXISTS Book;
CREATE TABLE Book   (book_id INTEGER PRIMARY KEY, title TEXT NOT NULL, author TEXT, price REAL CHECK (price > 0), copies INTEGER DEFAULT 1);
CREATE TABLE Member (member_id INTEGER PRIMARY KEY, name TEXT NOT NULL, joined TEXT);
CREATE TABLE Issue  (issue_id INTEGER PRIMARY KEY AUTOINCREMENT,
                     book_id INTEGER NOT NULL REFERENCES Book(book_id) ON DELETE CASCADE,
                     member_id INTEGER NOT NULL REFERENCES Member(member_id),
                     issue_date TEXT, returned INTEGER DEFAULT 0);
""")

def table(sql, params=()):
    print(pd.read_sql(sql, con, params=params).to_string(index=False), "\n")

# ---- CREATE
cur.executemany("INSERT INTO Book (title, author, price, copies) VALUES (?,?,?,?)", [
    ("Database Systems", "Elmasri", 650.0, 4), ("Python Basics", "Matthes", 480.0, 6), ("Data Pipelines", "Reis", 720.0, 2)])
cur.executemany("INSERT INTO Member (name, joined) VALUES (?,?)", [("Anaya", "2026-01-10"), ("Veer", "2026-02-01")])
cur.executemany("INSERT INTO Issue (book_id, member_id, issue_date) VALUES (?,?,?)", [(1, 1, "2026-03-01"), (2, 1, "2026-03-02"), (3, 2, "2026-03-03")])
con.commit()
print("CREATE -> books:"); table("SELECT * FROM Book")

# ---- READ
print("READ -> books issued per member:")
table("""SELECT m.name AS member, COUNT(*) AS books_issued, GROUP_CONCAT(b.title, ' | ') AS titles
         FROM Issue i JOIN Member m USING (member_id) JOIN Book b USING (book_id) GROUP BY m.name""")
print("READ -> books costing more than 500:")
table("SELECT title, price FROM Book WHERE price > ? ORDER BY price DESC", (500,))

# ---- UPDATE
cur.execute("UPDATE Book SET copies = copies - 1 WHERE book_id = 1")
cur.execute("UPDATE Issue SET returned = 1 WHERE issue_id = 2")
con.commit()
print("UPDATE -> stock and returns:")
table("SELECT book_id, title, copies FROM Book")
table("SELECT issue_id, book_id, returned FROM Issue")

# ---- DELETE (cascades to the Issue table)
cur.execute("DELETE FROM Book WHERE book_id = 3")
con.commit()
print("DELETE -> remaining books and issues:")
table("SELECT book_id, title FROM Book")
table("SELECT issue_id, book_id, member_id FROM Issue")
con.close()

Name: Ayushman Rajeev Verma
CREATE -> books:
 book_id            title  author  price  copies
       1 Database Systems Elmasri  650.0       4
       2    Python Basics Matthes  480.0       6
       3   Data Pipelines    Reis  720.0       2 

READ -> books issued per member:
member  books_issued                           titles
 Anaya             2 Database Systems | Python Basics
  Veer             1                   Data Pipelines 

READ -> books costing more than 500:
           title  price
  Data Pipelines  720.0
Database Systems  650.0 

UPDATE -> stock and returns:
 book_id            title  copies
       1 Database Systems       3
       2    Python Basics       6
       3   Data Pipelines       2 

 issue_id  book_id  returned
        1        1         0
        2        2         1
        3        3         0 

DELETE -> remaining books and issues:
 book_id            title
       1 Database Systems
       2    Python Basics 

 issue_id  book_id  member_id
        1       

**Conclusion:** Employee data stored in five formats was loaded into one DataFrame and the same checker class found the missing values, the duplicate record and the impossible salary and age values in every format. A custom binary file format was written, read and modified with struct, a web-server log was analysed with regular expressions, and CRUD operations were performed on a relational library database.